# Celabot — Paso 4: detección de robo armado

Caso #1 del plan, severidad **crítica**, latencia objetivo **< 5 s**. Es la razón principal por la que un dueño de tienda en Colombia paga por Celabot.

## Por qué "manos arriba" en vez de "arma visible"

Suena contraintuitivo pero es deliberado:

- Un detector de arma genérico (sin fine-tune) tiene **muchos falsos positivos**: cualquier objeto oscuro alargado se parece a una pistola.
- "Manos arriba sostenido" es **mucho más específico** y robusto a ángulos/luz.
- Usa el mismo YOLO-pose del paso 3 — cero costo de modelo nuevo.
- El VLM filtra las situaciones que sí parecen manos arriba pero no son robo (gimnasio, celebración, estiramiento, alcanzar algo alto).

El detector de arma fine-tuneado entra **como señal adicional que eleva prioridad**, no como gatillo único. Lo dejamos como stub al final.

## 0. Setup

Las mismas dependencias del paso 3.

In [ ]:
!pip install -q ultralytics supervision google-genai opencv-python-headless matplotlib

## 1. Sube tu video

Para que dispare en este prototipo necesitas un video donde **se vean manos arriba sostenidas**. Opciones:

- Te grabas tú simulando: ponte de pie frente a la cámara, levanta ambas manos por encima de la cabeza, mantenlas ahí 3–5 s, bájalas.
- Mejor aún si hay dos personas y ambas levantan las manos al mismo tiempo (la señal se vuelve mucho más fuerte).
- Si tienes un video real de cámara CCTV con un atraco simulado o real (con permisos), úsalo.

Cualquier resolución sirve, ideal ≥ 480p, cámara fija.

In [ ]:
from google.colab import files
import cv2

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'Video: {VIDEO_PATH}  {W}x{H}  fps={FPS:.1f}  dur={TOTAL_FRAMES/FPS:.1f}s')

## 2. La señal "manos arriba"

Por cada persona y cada frame definimos un estado discreto:

- `'both'` — ambas muñecas claramente arriba de la nariz.
- `'one'` — una sola muñeca arriba.
- `'none'` — ninguna arriba.
- `None` — no calculable (nariz u hombros no visibles con suficiente confianza).

**Por qué la nariz como referencia y no los hombros**: en un atraco las manos suelen estar por encima de la cabeza, no a la altura del hombro. Usamos `wrist_y < nose_y - margen` donde `margen = 0.3 × distancia(nariz, hombros)`. Esto evita confundir con "señalar al frente" o "saludar".

Nota sobre el sistema de coordenadas: en imagen, **menor `y` = más arriba**.

In [ ]:
import numpy as np

KP_NOSE = 0
KP_LSHOULDER, KP_RSHOULDER = 5, 6
KP_LWRIST, KP_RWRIST = 9, 10
KP_CONF_MIN = 0.3

def hands_up_state(kp_xy, kp_conf):
    """Devuelve 'both' | 'one' | 'none' | None."""
    needed = [KP_NOSE, KP_LSHOULDER, KP_RSHOULDER]
    if not all(kp_conf[k] >= KP_CONF_MIN for k in needed):
        return None
    nose_y = kp_xy[KP_NOSE, 1]
    shoulder_y = (kp_xy[KP_LSHOULDER, 1] + kp_xy[KP_RSHOULDER, 1]) / 2
    head_size = max(abs(shoulder_y - nose_y), 1)
    threshold_y = nose_y - 0.3 * head_size  # arriba de la nariz con margen

    def above(wrist_idx):
        if kp_conf[wrist_idx] < KP_CONF_MIN:
            return None
        return bool(kp_xy[wrist_idx, 1] < threshold_y)

    l_up = above(KP_LWRIST)
    r_up = above(KP_RWRIST)
    visible = [x for x in (l_up, r_up) if x is not None]
    if not visible:
        return None
    ups = sum(1 for x in visible if x)
    if ups == 2:
        return 'both'
    if ups == 1:
        return 'one'
    return 'none'

## 3. Carril rápido: pose + tracking + state machine de manos arriba

Por cada track mantenemos un "intervalo activo" mientras el estado sea `both` o `one`. Cuando vuelve a `none` (o se pierde el track), cerramos el intervalo si duró al menos `HANDS_UP_MIN_DURATION_S`.

Visualización en el video anotado:
- Esqueleto normal por defecto.
- **Círculos amarillos** sobre las muñecas si están arriba (state `one`).
- **Círculos rojos + texto `HANDS UP`** si las dos están arriba (state `both`).
- Barra superior con el conteo de personas con manos arriba en el frame actual.

In [ ]:
from ultralytics import YOLO
import supervision as sv

pose_model = YOLO('yolo11n-pose.pt')

HANDS_UP_MIN_DURATION_S = 1.5

hands_up_intervals: dict[int, list[dict]] = {}
hands_up_active: dict[int, dict] = {}

def open_or_extend(tid, ts, state):
    if tid not in hands_up_active:
        hands_up_active[tid] = {'start': ts, 'last': ts, 'max_state': state}
    else:
        hands_up_active[tid]['last'] = ts
        if state == 'both':
            hands_up_active[tid]['max_state'] = 'both'

def close_if_long(tid):
    rec = hands_up_active.pop(tid, None)
    if rec and rec['last'] - rec['start'] >= HANDS_UP_MIN_DURATION_S:
        hands_up_intervals.setdefault(tid, []).append({
            'start': rec['start'], 'end': rec['last'], 'max_state': rec['max_state']
        })

def callback(frame, frame_idx):
    ts = frame_idx / FPS
    results = pose_model.track(frame, persist=True, classes=[0], tracker='bytetrack.yaml', verbose=False)[0]
    annotated = results.plot()

    if results.boxes is None or results.boxes.id is None or results.keypoints is None:
        return annotated

    ids = results.boxes.id.cpu().numpy().astype(int)
    kps_xy = results.keypoints.xy.cpu().numpy()
    kps_conf = (results.keypoints.conf.cpu().numpy() if results.keypoints.conf is not None
                else np.ones((kps_xy.shape[0], 17)))
    boxes = results.boxes.xyxy.cpu().numpy()

    seen_now = set()
    n_hands_up_frame = 0
    for i, tid in enumerate(ids):
        tid = int(tid)
        seen_now.add(tid)
        state = hands_up_state(kps_xy[i], kps_conf[i])
        if state in ('both', 'one'):
            open_or_extend(tid, ts, state)
            n_hands_up_frame += 1
            color = (0, 0, 255) if state == 'both' else (0, 255, 255)
            # marcar muñecas visibles arriba
            for kp_idx in (KP_LWRIST, KP_RWRIST):
                if kps_conf[i, kp_idx] >= KP_CONF_MIN:
                    x, y = int(kps_xy[i, kp_idx, 0]), int(kps_xy[i, kp_idx, 1])
                    cv2.circle(annotated, (x, y), 12, color, -1)
            if state == 'both':
                x1, y1, _, _ = boxes[i].astype(int)
                cv2.putText(annotated, 'HANDS UP', (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        else:
            close_if_long(tid)

    # Tracks que ya no se ven en este frame: cerrar
    for tid in list(hands_up_active.keys()):
        if tid not in seen_now:
            close_if_long(tid)

    # Banner superior con conteo
    if n_hands_up_frame > 0:
        cv2.rectangle(annotated, (0, 0), (W, 35), (0, 0, 0), -1)
        cv2.putText(annotated, f'Manos arriba: {n_hands_up_frame}', (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return annotated

print('Procesando video con pose + tracking + state machine...')
sv.process_video(source_path=VIDEO_PATH, target_path='annotated_raw.mp4', callback=callback)
# Cerrar intervalos abiertos al final del video
for tid in list(hands_up_active.keys()):
    close_if_long(tid)

total_intervals = sum(len(v) for v in hands_up_intervals.values())
print(f'Tracks con intervalos: {len(hands_up_intervals)} | Intervalos totales: {total_intervals}')
for tid, ivs in hands_up_intervals.items():
    for iv in ivs:
        print(f"  tid #{tid}: {iv['start']:.1f}s -> {iv['end']:.1f}s ({iv['end']-iv['start']:.1f}s) max={iv['max_state']}")

## 4. Mira el video anotado

In [ ]:
import subprocess
from IPython.display import HTML
from base64 import b64encode

subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', 'annotated_raw.mp4',
    '-c:v', 'libx264', '-preset', 'ultrafast', '-movflags', '+faststart', '-an',
    'annotated.mp4',
], check=True)
mp4 = open('annotated.mp4', 'rb').read()
HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

## 5. De intervalos a candidatos priorizados

Cada intervalo de manos arriba es un candidato. Le agregamos dos metadatos clave para que el carril lento sepa a qué prestarle atención primero:

- **`concurrent_at_start`**: cuántas personas distintas tenían manos arriba al mismo tiempo que esta. Si son ≥ 2, la señal es mucho más fuerte (un atraco típico tiene varias personas con manos arriba al mismo tiempo).
- **`max_state`**: si en algún momento del intervalo la persona tuvo ambas manos arriba (más fuerte que solo una).

Ordenamos los candidatos por `concurrent_at_start` desc y duración desc.

In [ ]:
def concurrent_at(ts):
    count = 0
    for tid, ivs in hands_up_intervals.items():
        for iv in ivs:
            if iv['start'] <= ts <= iv['end']:
                count += 1
    return count

armed_candidates = []
for tid, ivs in hands_up_intervals.items():
    for iv in ivs:
        armed_candidates.append({
            'tid': tid,
            'start': max(0, iv['start'] - 1),
            'end': iv['end'] + 1,
            'duration': iv['end'] - iv['start'],
            'max_state': iv['max_state'],
            'concurrent_at_start': concurrent_at(iv['start']),
        })

armed_candidates.sort(
    key=lambda c: (c['concurrent_at_start'], c['duration']), reverse=True
)

print(f'Candidatos de robo armado: {len(armed_candidates)}')
for c in armed_candidates[:5]:
    print(f"  tid #{c['tid']} {c['start']:.1f}-{c['end']:.1f}s "
          f"({c['duration']:.1f}s, max={c['max_state']}, concurrentes={c['concurrent_at_start']})")

## 6. Carril lento: Gemini confirma robo armado

El *prompt* es el más crítico de todos los que hemos escrito porque las consecuencias son altas (alerta a policía, susto al dueño). Enumeramos explícitamente los falsos positivos comunes y damos pesos al contexto.

Para el prototipo confirmamos solo el candidato top (mayor concurrencia / duración). En producción confirmarías los top-K en paralelo con timeout duro de 3 s.

In [ ]:
import os, subprocess, time, json, getpass
from google import genai

def extract_clip(src, start, end, out):
    if os.path.exists(out):
        os.remove(out)
    duration = max(0.5, end - start)
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', src, '-ss', f'{start:.2f}', '-t', f'{duration:.2f}',
        '-c:v', 'libx264', '-preset', 'ultrafast', '-movflags', '+faststart', '-an',
        out,
    ], check=True)
    return out

def ask_gemini(client, clip_path, prompt, model='gemini-2.5-flash'):
    vf = client.files.upload(file=clip_path)
    while vf.state.name == 'PROCESSING':
        time.sleep(2)
        vf = client.files.get(name=vf.name)
    resp = client.models.generate_content(model=model, contents=[vf, prompt])
    raw = resp.text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

GEMINI_API_KEY = getpass.getpass('Pega tu API key de Gemini: ')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
PROMPT_ARMED = '''Eres un analista de seguridad de tienda en Colombia. Una heurística de pose detectó {n_people} persona(s) con las manos arriba sostenido por {duration:.1f} segundos. Esto puede indicar un ROBO ARMADO en curso.

Mira el clip y responde ESTRICTAMENTE en JSON:
{{"anomaly": bool, "confidence": float, "category": "robo_armado" | "celebracion" | "ejercicio" | "alcanzar_alto" | "normal", "observed_behaviors": [str], "weapon_visible": bool, "victims_count": int, "reason": str, "suggested_action": str}}

Reglas para diferenciar:
- ROBO ARMADO: una o más personas con manos arriba en postura tensa/rígida; otra persona con postura amenazante (brazo extendido, posible arma); contexto de tienda con caja/mostrador; las víctimas miran al asaltante; pueden estar de rodillas.
- NO es robo:
  - Celebración deportiva/musical: gente saltando, abrazándose, sonriendo, gestos amplios.
  - Ejercicio (yoga, gimnasio, baile): postura controlada, ropa deportiva, sin tensión.
  - Niños jugando: tamaños distintos, juego activo.
  - Alcanzar algo alto: estiramiento puntual, una sola mano, mirada hacia arriba.
  - Estirarse al despertar / al levantarse.
- Si NO es claramente robo, anomaly=false con confidence baja.
- weapon_visible=true SOLO si claramente identificas un arma de fuego o blanca; en duda, false.
- victims_count = cuántas personas distintas con manos arriba en postura de víctima.
- Solo el JSON, sin texto extra.'''

if not armed_candidates:
    print('Sin candidatos de manos arriba sostenido. Prueba con un video donde levantes claramente ambas manos por encima de la cabeza al menos 2s.')
else:
    top = armed_candidates[0]
    print(f"\n=== Confirmando candidato top: tid #{top['tid']} duración {top['duration']:.1f}s concurrentes {top['concurrent_at_start']} ===")
    end = min(top['end'], top['start'] + 12)
    extract_clip(VIDEO_PATH, top['start'], end, 'clip_armed.mp4')
    prompt = PROMPT_ARMED.format(
        n_people=top['concurrent_at_start'],
        duration=top['duration'],
    )
    try:
        verdict = ask_gemini(client, 'clip_armed.mp4', prompt)
        print(json.dumps(verdict, indent=2, ensure_ascii=False))
        if verdict.get('anomaly') and verdict.get('confidence', 0) >= 0.6:
            print('\n🚨 ALERTA: en producción esto dispararía notificación WhatsApp + push + voz al dueño y, opcionalmente, a empresa de seguridad/123.')
    except Exception as e:
        print(f'Error: {e}')

## 7. Stub: integración futura del detector de arma

Cuando tengamos un YOLO11 fine-tuneado para detección de arma (paso siguiente), se enchufa con esta interfaz mínima. La lógica de fusión queda igual: el detector eleva la prioridad o, si la confianza es muy alta, dispara aunque no haya "manos arriba".

Fuentes para entrenamiento o pesos pre-entrenados:

- **Roboflow Universe** (busca "weapon", "gun", "handgun"): pesos descargables, formatos YOLO listos.
- **Sohas dataset**: clásico para fine-tuning de arma blanca + de fuego.
- **Hugging Face Hub**: hay varios `weapon-detection-yolov8` publicados.
- **Mezcla con sintéticos** (Unreal/Unity + asset packs): para subir recall en escenarios raros como atraco a tienda.

Interfaz objetivo (no se ejecuta sin pesos):

```python
def detect_weapon(frame: np.ndarray) -> list[dict]:
    """
    Returns a list of detections: 
    [{'box': [x1,y1,x2,y2], 'conf': float, 'class': 'gun'|'knife'}]
    """
    results = weapon_model(frame, verbose=False)[0]
    out = []
    for b in results.boxes:
        out.append({
            'box': b.xyxy[0].tolist(),
            'conf': float(b.conf[0]),
            'class': results.names[int(b.cls[0])],
        })
    return out
```

Y en el callback:

```python
weapons = detect_weapon(frame)
if weapons:
    # Candidato de PRIORIDAD MÁXIMA, dispara aunque no haya manos arriba
    # Anota el frame con cajas naranjas grandes y "WEAPON"
    ...
```

Mantener este detector aislado tras una interfaz pequeña es exactamente lo que pide el plan §4.3 (capa de abstracción Detector/Segmenter/Reasoner).

## 8. Limitaciones honestas

1. **Cobertura de pose**: cámaras montadas muy alto o muy bajo distorsionan el reconocimiento de muñecas/nariz. Mejor a 2.5–3 m de altura, ángulo 15–25° hacia abajo.
2. **Falsos positivos legítimos**: gimnasios, peluquerías (manos arriba sobre la cabeza), bares en celebraciones, niños jugando. El VLM filtra la mayoría, pero el dueño puede definir "esta tienda es un bar, ignora celebración".
3. **Atraco silencioso/encañonado por detrás**: si las víctimas no levantan las manos (ej. mostrador desde atrás), la heurística no dispara. Esto es donde un detector de arma + comportamiento atípico (gente quieta, brazo extendido) entra en paso 5.
4. **Pocos frames del incidente**: si el robo dura 6 s y nuestro `HANDS_UP_MIN_DURATION_S=1.5`, vamos a disparar; pero si por la cámara solo se ven 1.2 s claros, perdemos. Mitigación: bajar el umbral por tienda según calibración.
5. **Latencia del VLM**: Gemini puede tomar 3–8 s en confirmar. Cumple el SLA de 5 s del plan al borde. Para producción real: VLM con *prompt caching*, modelo más pequeño (Flash), o destilar a modelo propio.

## Implicaciones para el plan

Con este enfoque, sugiero ajustar el §11 del plan:

- **Fase 0 piloto**: solo robo armado (manos arriba + VLM) + intrusión fuera de horario. Dos casos críticos, alta percepción de valor.
- **Fase 1 MVP**: lo anterior + detector de arma fine-tuneado + grab-and-run con cruce de salida (paso 2). Mantener concealment para Fase 2.
- **Fase 2**: audio (gunshot, gritos) — el complemento natural de robo armado.
- **Marketing**: "Celabot detecta el robo armado en segundos". Un solo titular vendedor.

## Siguiente paso

Tres caminos para el **paso 5**:

1. **Fine-tune real del detector de arma** con un dataset público (Roboflow Universe o Sohas). Notebook que descarga, entrena 30 epochs en T4, guarda pesos y los enchufa al `detect_weapon` stub.
2. **RTSP en vivo** con mediaMTX — del MP4 al stream real, medir latencia end-to-end del pipeline completo (incluyendo VLM).
3. **Audio gunshot** — bajar PANNs/YAMNet preentrenado y combinar disparos detectados con manos arriba para señales fusionadas.

Dime cuál.